# F2 · GridSearchCV, RandomizedSearchCV and HalvingRandomSearchCV compared

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Compare three hyperparameter search strategies for a gradient boosting by number of trainings, time, validation score and AUC on the
test set, and understand when random search and halving save effort without losing quality.

## Dataset

`ds.loans(with_missing=False)`: the four numeric columns and the target `defaulted`, 1500 rows (stratified 75/25 split). Model:
`GradientBoostingClassifier` with a search space on `learning_rate`, `max_depth`, `n_estimators` and `subsample` (54 combinations).

## Educational purpose

Know what you buy with each strategy: the grid is exhaustive but costly, random search explores a lot with little, halving throws away poor
combinations early using little data at the start. The comparison is done on the numbers.

## Task

1. Build three searches with 3 folds each and `scoring="roc_auc"`: `GridSearchCV` on the full grid (`grid`), `RandomizedSearchCV` with `n_iter=12` and `random_state=0` on the same space, `HalvingRandomSearchCV` with `n_candidates=27`, `factor=3`, `random_state=0`.
2. For each one measure the time with `time.time()` (seconds) while you run `fit(X_tr, y_tr)`.
3. Fill `table` (rows: `grid`, `random`, `halving`; columns `n_trainings` (number of `fit` = rows of `cv_results_` × 3 folds; for halving, sum over the iterations: use `len(cv_results_["params"]) * 3`), `time_s`, `best_score`, `test_auc`).
4. Save `random_saving = 1 − n_trainings(random)/n_trainings(grid)` and `random_loss = test_auc(grid) − test_auc(random)`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.experimental import enable_halving_search_cv              # noqa: F401
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, HalvingRandomSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]]
y = df["defaulted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(3, shuffle=True, random_state=0)
est = GradientBoostingClassifier(random_state=0)
grid = {"learning_rate": [0.03, 0.1, 0.3], "max_depth": [2, 3, 4], "n_estimators": [50, 100, 200], "subsample": [0.6, 1.0]}

## Your work

In [ ]:
searches = {
    "grid": GridSearchCV(est, grid, cv=cv, scoring="roc_auc"),
    "random": ...,
    "halving": ...,
}
table = pd.DataFrame(index=list(searches), columns=["n_trainings", "time_s", "best_score", "test_auc"], dtype=float)
for name, search in searches.items():
    ...
table.round(3)

In [ ]:
random_saving = ...
random_loss = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`RandomizedSearchCV(est, grid, n_iter=12, cv=cv, scoring="roc_auc", random_state=0)`; `HalvingRandomSearchCV(est, grid, n_candidates=27, factor=3, cv=cv, scoring="roc_auc", random_state=0)`. Loop: `t0 = time.time(); search.fit(X_tr, y_tr); elapsed = time.time() - t0`; `len(search.cv_results_["params"]) * 3`.'):
    check(set(table.index) == {"grid", "random", "halving"} and table.notna().all().all(), "complete table", "three rows and four columns with no NaN")
    check(table.loc["grid", "n_trainings"] == 54 * 3, "the grid does 54 x 3 = 162 trainings", "54 combinations x 3 folds")
    check(table.loc["random", "n_trainings"] == 12 * 3, "random search does 12 x 3 = 36", "12 combinations x 3 folds")
    check(table.loc["random", "time_s"] < table.loc["grid", "time_s"] / 2, "random search costs less than half of the grid in time", "36 trainings against 162")
    check(random_saving > 0.7 and abs(random_loss) < 0.03, "random search saves over 70% of the trainings without losing more than 0.03 of test AUC",
          "compare n_trainings and test_auc between grid and random")
    check(table.loc["halving", "n_trainings"] > table.loc["random", "n_trainings"] and table["test_auc"].min() > 0.72,
          "halving tries more combinations than random search (but on less data for the first ones), and all the strategies have a decent test AUC",
          "halving starts with 27 candidates and reduces them at every iteration")
done()

---
The commented solution is in `solutions/F_compare/F2_grid_random_halving_compared.ipynb`. Open it after you pass the check, or after you have really tried.